## Phase 2 – Model Training


In this phase, we focused on training different machine learning models using the preprocessed data from Phase 1.  
The main goal was to prepare and train our models so we can later evaluate them and decide which one works best for detecting the spam emails.

We decided to test three models that are commonly used and well-known for text classification:

1. **Logistic Regression** – a simple but powerful baseline model that usually performs very well with TF-IDF text data.  
2. **Linear SVM (Support Vector Machine)** – works great with high-dimensional features and can handle TF-IDF vectors efficiently.  
3. **Naive Bayes (optional)** – a lightweight model that trains quickly and provides a good point of comparison for more complex algorithms.

We chose these models because they are effective, easy to interpret, and well-suited for spam or phishing detection tasks.


## 📋 Work Distribution Table

| Step | Task Description | Assigned To | Subtasks / Notes |
|------|------------------|--------------|------------------|
| **1** | **Data Preparation & Notebook Review** | **Renad** | • Split data into **train** and **testing** sets.<br>• Review data preprocessing notebook, **Conclusion: Findings and diagnostic analysis over the model evaluation** .<br>• Suggest suitable **models** for training.|
| **2** | **Model Selection & Training** | **Salha & Rawan** | • Pick model.<br>• Train the model using **train** and **eval** sets.<br>• Record and document **training metrics**. |
| **3** | **Model Evaluation** | **Rahaf** | • Evaluate the final model using the **testing** set.<br>• Generate **performance metrics** (Precision, Recall, F1, Confusion Matrix, etc.).<br>• Summarize and interpret the results. |
| **4** | **Feature Importance Analysis** | **Fajr** | • Determine **feature importance** using the trained model.<br>• Create visualizations (e.g., SHAP, bar charts).<br>• Write an **insight summary** explaining feature impact. |


### Data Loading and Splitting

Now we start by importing our data and splitting it into training and testing sets.    
We’re using the same data split for all models so that every model is trained and tested on identical data.      
This way, later when we compare their performance, the results will be fair and consistent.

In [1]:
import joblib
import numpy as np

# Load preprocessed train/test features
data = joblib.load("artifacts/feature_data.joblib")

X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

print("Data loaded successfully")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


# Drop 'receiver_spam_ratio' from numeric features


# Your numeric columns from before
numeric_cols = [
    'urls', 'receiver_spam_ratio', 'hour',
    'capital_letter_count', 'capital_ratio',
    'exclamation_count', 'question_count', 'special_char_count',
    'day_of_week_Friday', 'day_of_week_Monday',
    'day_of_week_Saturday', 'day_of_week_Sunday',
    'day_of_week_Thursday', 'day_of_week_Tuesday',
    'day_of_week_Wednesday'
]

# Index of the feature to drop
drop_idx = numeric_cols.index('receiver_spam_ratio')

# Last N columns are numeric
num_cols_start = X_train.shape[1] - len(numeric_cols)

# Compute absolute index in sparse matrix
abs_idx = num_cols_start + drop_idx

# Drop column from train and test
X_train = X_train[:, np.arange(X_train.shape[1]) != abs_idx]
X_test = X_test[:, np.arange(X_test.shape[1]) != abs_idx]

print("Dropped 'receiver_spam_ratio'")
print("New X_train shape:", X_train.shape)
print("New X_test shape:", X_test.shape)


Data loaded successfully
X_train shape: (31311, 5015)
X_test shape: (7828, 5015)
y_train shape: (31311,)
y_test shape: (7828,)
Dropped 'receiver_spam_ratio'
New X_train shape: (31311, 5014)
New X_test shape: (7828, 5014)


### Logistic Regression

We started with Logistic Regression because it’s often a strong baseline model for binary classification.  
It’s simple, fast, and works really well with large TF-IDF datasets.  
Here, we train it and show the results just to confirm that it’s working —  
We’ll explain these results in more detail later in the evaluation section of this phase.

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

# Create the Logistic Regression model
log_model = LogisticRegression(max_iter=2000, random_state=42)

# Define hyperparameter values to try
param_grid = {
    'C': [0.01, 0.1, 1, 10],       # Regularization strength
    'solver': ['liblinear', 'saga'] # Different solvers
}

# Set up GridSearchCV with 3-fold cross-validation
grid_log = GridSearchCV(log_model, param_grid, cv=3, scoring='accuracy', n_jobs=-1)

# Print training start message
print("Training Logistic Regression")  

# Train the model using Grid Search
grid_log.fit(X_train, y_train)
print("Model trained successfully")  # Keep the post-training message

# Print best parameters found during search
print("Best parameters:", grid_log.best_params_)
print("Best CV score:", round(grid_log.best_score_ * 100, 2), "%")

# Test the model on new/unseen data
y_pred = grid_log.predict(X_test)

# Compute accuracy and print classification report
acc = accuracy_score(y_test, y_pred)
print("Test Accuracy:", round(acc * 100, 2), "%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Training Logistic Regression
Model trained successfully
Best parameters: {'C': 10, 'solver': 'liblinear'}
Best CV score: 99.45 %
Test Accuracy: 99.74 %

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3462
           1       1.00      1.00      1.00      4366

    accuracy                           1.00      7828
   macro avg       1.00      1.00      1.00      7828
weighted avg       1.00      1.00      1.00      7828



### Linear SVM (Support Vector Machine)

Next, we trained a Linear SVM model.  
This model is great for high-dimensional data like our TF-IDF features.  
We also used Grid Search with Cross Validation to find the best hyperparameter “C” value and to make sure we don’t overfit.  
Again, we’re only checking that it trains and runs properly — we’ll analyze its performance later.

In [3]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# define our SVM model and parameter grid
param_grid = {'C': [0.1, 1, 10]}
svm = LinearSVC(max_iter=2000, random_state=42)

# use GridSearchCV to find the best C value
grid = GridSearchCV(svm, param_grid, cv=3, n_jobs=-1)
print("Training and tuning Linear SVM...")
grid.fit(X_train, y_train)

# print the best parameters and validation score
print("Best parameters:", grid.best_params_)
print("Best CV score:", round(grid.best_score_ * 100, 2), "%")

# test on unseen data
y_pred_svm = grid.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print("Test Accuracy:", round(acc_svm * 100, 2), "%")

# quick report just to confirm it’s working
print("\n for verification only We’ll explain results in detail later in the evaluation section\n")
print(classification_report(y_test, y_pred_svm))



Training and tuning Linear SVM...
Best parameters: {'C': 1}
Best CV score: 99.5 %
Test Accuracy: 99.76 %

 for verification only We’ll explain results in detail later in the evaluation section

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3462
           1       1.00      1.00      1.00      4366

    accuracy                           1.00      7828
   macro avg       1.00      1.00      1.00      7828
weighted avg       1.00      1.00      1.00      7828



### Naive Bayes 

Lastly, we trained a Naive Bayes model as one more way to test our data.
It’s a simple and fast algorithm that uses probabilities to decide whether an email is phishing or not.
Even though it’s not as complex as the other models, it’s still helpful for seeing how a quick, lightweight method performs in comparison.

In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

# Create Naive Bayes model
nb = MultinomialNB()

# Print training start message
print("Training Naive Bayes...")

# Define Grid Search with hyperparameter alpha
param_grid_nb = {'alpha': [0.1, 0.5, 1.0, 2.0]}

# Set up GridSearchCV with 3-fold cross-validation
grid_nb = GridSearchCV(nb, param_grid_nb, cv=3, scoring='accuracy', n_jobs=-1)

# Train the model using Grid Search
grid_nb.fit(X_train, y_train)
print("Model trained successfully!")

# Print best parameters found during search
print("Best parameters:", grid_nb.best_params_)
print("Best CV score:", round(grid_nb.best_score_ * 100, 2), "%")

# Test the model on unseen data
y_pred_nb = grid_nb.predict(X_test)

# Compute accuracy and print classification report
acc_nb = accuracy_score(y_test, y_pred_nb)
print("Test Accuracy:", round(acc_nb * 100, 2), "%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))


Training Naive Bayes...
Model trained successfully!
Best parameters: {'alpha': 2.0}
Best CV score: 96.59 %
Test Accuracy: 96.35 %

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.99      0.96      3462
           1       0.99      0.94      0.97      4366

    accuracy                           0.96      7828
   macro avg       0.96      0.97      0.96      7828
weighted avg       0.97      0.96      0.96      7828



### Model Evaluation  

After training all three models  Logistic Regression, Linear SVM, and Naive Bayes 
this section focuses on evaluating and comparing using model evaluation metrics

The main objective here is to determine which model performs best overall and is therefore the most suitable for phishing email detection.
We use standard classification metrics Accuracy, Precision, Recall, and F1-score to ensure that our evaluation captures every important aspect of model performance, including reliability and consistency on unseen data.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Compute evaluation metrics for all models
metrics = {
    "Model": ["Logistic Regression", "Linear SVM", "Naive Bayes"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_svm),
        accuracy_score(y_test, y_pred_nb)
    ],
    "Precision": [
        precision_score(y_test, y_pred, average="macro"),
        precision_score(y_test, y_pred_svm, average="macro"),
        precision_score(y_test, y_pred_nb, average="macro")
    ],
    "Recall": [
        recall_score(y_test, y_pred, average="macro"),
        recall_score(y_test, y_pred_svm, average="macro"),
        recall_score(y_test, y_pred_nb, average="macro")
    ],
    "F1-Score": [
        f1_score(y_test, y_pred, average="macro"),
        f1_score(y_test, y_pred_svm, average="macro"),
        f1_score(y_test, y_pred_nb, average="macro")
    ]
}

# Create a clean summary table
df_metrics = pd.DataFrame(metrics)
df_metrics[["Accuracy", "Precision", "Recall", "F1-Score"]] = (
    df_metrics[["Accuracy", "Precision", "Recall", "F1-Score"]].round(4)
)

print("Model Evaluation Results:\n")
display(df_metrics)


Model Evaluation Results:



,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.9974,0.9974,0.9974,0.9974
1,Linear SVM,0.9976,0.9976,0.9975,0.9975
2,Naive Bayes,0.9635,0.9617,0.9666,0.9632


### Metric Comparison
The most important metric is Recall, as it indicates how many actual phishing or spam  emails the model successfully identified.


If the Recall is low, it means the model is missing some phishing emails and that can be risky, since those missed emails could harm users or compromise security.
It’s better for the model to flag a few safe emails by mistake than to let a dangerous one pass through.
So, in our case, a higher Recall value means better protection and a more reliable system

In [6]:
# Create a clean summary table
df_metrics = pd.DataFrame(metrics)
df_metrics[["Accuracy", "Precision", "Recall", "F1-Score"]] = (
    df_metrics[["Accuracy", "Precision", "Recall", "F1-Score"]].round(4)
)

print(" Model Evaluation Results:\n")
display(df_metrics)


 Model Evaluation Results:



,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.9974,0.9974,0.9974,0.9974
1,Linear SVM,0.9976,0.9976,0.9975,0.9975
2,Naive Bayes,0.9635,0.9617,0.9666,0.9632


Both Logistic Regression and Linear SVM showed outstanding performance, achieving almost perfect scores across all evaluation metrics.
However, Linear SVM performed slightly better in every category including Accuracy, Precision, Recall, and F1-Score.
Even though the improvement is small, it still indicates that the Linear SVM model makes more correct predictions overall and generalizes a bit better to unseen data.

Since our main goal is to detect as many phishing emails as possible, Recall is the most important metric.
A higher Recall means the model can correctly identify more phishing emails without missing any — which is crucial for maintaining security and preventing real threats from slipping through.
Linear SVM achieved a slightly higher Recall (0.9983 vs 0.9976), showing it’s more sensitive in detecting phishing messages.


### Final Decision

Based on these results, Linear SVM was selected as the final model for phishing email detection.
It provides the best balance between accuracy, Recall, and reliability, making it the most suitable choice

### Coefficient-Based Feature Importance 

Since our final model is **Linear SVM**, we can understand what features had the most impact on its predictions.  
This type of model gives a weight to every feature, showing how much it pushes the prediction toward spam or not spam.

In **Phase 1**, we combined the email **subject** and **body** into one column called `combined_text` and used TF-IDF to turn the text into numeric features.  
Each of these features represents a word, and its weight in the model shows how important that word is.

Positive weights mean the word pushes the model toward **spam**, while negative weights push toward **not spam**.  
By looking at these weights, we can tell which words had the strongest influence on the model’s decisions.


In [7]:
import joblib
import numpy as np
import pandas as pd
from scipy.sparse import issparse

# Load vectorizer and preprocessed train/test data
vec = joblib.load("artifacts/tfidf_vectorizer.joblib")   # TF-IDF from Phase 1
data = joblib.load("artifacts/feature_data.joblib")      # contains train/test splits

X_train = data["X_train"]   # or X_test, depending on what you need
y_train = data["y_train"]

# Load your trained Linear SVM model
svm_best = grid.best_estimator_  # make sure this is already trained on X_train

# Get feature names from TF-IDF and number of text features
feature_names = np.array(vec.get_feature_names_out())
n_text = feature_names.shape[0]

# Get model coefficients and flatten them
coef = svm_best.coef_
if issparse(coef):
    coef = coef.toarray()
coef = coef.ravel()

# Focus only on text features (exclude numeric features)
coef_text = coef[:n_text]

# Top positive and negative weights
top_k = 20
pos_idx = np.argpartition(coef_text, -top_k)[-top_k:]
pos_idx = pos_idx[np.argsort(coef_text[pos_idx])[::-1]]

neg_idx = np.argpartition(coef_text, top_k)[:top_k]
neg_idx = neg_idx[np.argsort(coef_text[neg_idx])]

top_spam = pd.DataFrame({
    "feature": feature_names[pos_idx],
    "weight": coef_text[pos_idx]
}).reset_index(drop=True)

top_notspam = pd.DataFrame({
    "feature": feature_names[neg_idx],
    "weight": coef_text[neg_idx]
}).reset_index(drop=True)

print("Top features driving SPAM (positive weights):")
display(top_spam)

print("Top features driving NOT SPAM (negative weights):")
display(top_notspam)


Top features driving SPAM (positive weights):


,feature,weight
0,hash 20,1.676136
1,love,1.625805
2,custom alert,1.514661
3,sex,1.483369
4,payment,1.480585
5,20 hash,1.413874
6,girl,1.354261
7,utah,1.329521
8,daily 10,1.291595
9,bank,1.233596


Top features driving NOT SPAM (negative weights):


,feature,weight
0,fork,-3.447083
1,opensuse,-3.302422
2,ierant,-3.060701
3,wrote,-2.670929
4,ilug,-2.630998
5,uai,-2.511842
6,tony,-2.267662
7,spam,-2.151442
8,thanks,-2.105112
9,cheers,-1.788019


### Results and Observations 

From the results, the model gave higher positive weights to words like **“bank”**, **“payment”**, **“won”**, and **“debt”**.  
These are common words in spam emails, especially the ones related to money or urgent actions, so they pushed the model toward predicting spam.

On the other hand, words like **“thanks”**, **“newsletter”**, **“people”**, and **“date”** had negative weights, meaning they usually appear in normal or safe emails.  
So when the model sees these words, it predicts not spam.

Overall, this shows that the Linear SVM successfully learned real spam patterns.
It focused on financial or alert-related words and ignored casual or harmless ones, which explains why it achieved such a high recall.



### Conclusion: Findings and diagnostic analysis over the model evaluation
All models achieved accuracy scores between 99% and 100%, which at first seemed perfect. However, this level of performance is suspiciously high and indicates a case of data leakage.

In the preprocessing phase, the TF-IDF vectorizer and the scaler were fitted on the entire dataset (df) before splitting into train and test sets. This caused the models to “see” information from the test data during training, allowing them to memorize patterns instead of truly generalizing.

I detected this issue because **precision and recall values were nearly identical and extremely high**, which is unusual — in most realistic scenarios, these metrics tend to vary or trade off against each other.

### Results with Data Leak

| Model                | Accuracy | Precision | Recall  | F1-Score |
|----------------------|---------|-----------|--------|----------|
| Logistic Regression  | 0.9976  | 0.9976    | 0.9976 | 0.9976   |
| Linear SVM           | 0.9983  | 0.9983    | 0.9983 | 0.9983   |
| Naive Bayes          | 0.9943  | 0.9943    | 0.9943 | 0.9943   |


### small fix: lets edit the TFIDF step in the preprocess

we did fit trnasfrom the data before splitting then we fixed it and re ran the code 

### Model Evaluation Results after fixing the data leak issue

| Model                | Accuracy | Precision | Recall  | F1-Score |
|----------------------|---------|-----------|--------|----------|
| Logistic Regression  | 0.9976  | 0.9976    | 0.9976 | 0.9976   |
| Linear SVM           | 0.9982  | 0.9982    | 0.9982 | 0.9982   |
| Naive Bayes          | 0.9941  | 0.9941    | 0.9941 | 0.9941   |



### Why still simialr precision and recall ? 
lets check all possible causes

In [8]:
#1.Class balance
y_train.value_counts(normalize=True)



label
1    0.557663
0    0.442337
Name: proportion, dtype: float64

#### The classes actually pretty balanced, so class imbalance isn’t the reason your precision ≈ recall.

### The dataset is genuinely clear?

Those numeric features (like receiver_spam_ratio, capital_ratio, urls, etc.) can be very strong discriminators between spam and non-spam.
If a few of them correlate almost perfectly with the label, the models will all lock on to the same obvious signal →  precision ≈ recall ≈ accuracy ≈ F1 ≈ 99%.

### Lets check the corroliation

In [9]:
type(X_train)


scipy.sparse._csr.csr_matrix

In [10]:
#2. high corrliation features
import joblib
import pandas as pd

#  Load preprocessed features
data = joblib.load("artifacts/feature_data.joblib")
X_train = data["X_train"]
y_train = data["y_train"]

# These are the numeric columns 
numeric_cols = [
    'urls', 'receiver_spam_ratio', 'hour',
    'capital_letter_count', 'capital_ratio',
    'exclamation_count', 'question_count', 'special_char_count',
    'day_of_week_Friday', 'day_of_week_Monday',
    'day_of_week_Saturday', 'day_of_week_Sunday',
    'day_of_week_Thursday', 'day_of_week_Tuesday',
    'day_of_week_Wednesday'
]

# Extract numeric part from sparse matrix
num_features = X_train[:, -len(numeric_cols):].toarray()  # last columns are numeric
df_num = pd.DataFrame(num_features, columns=numeric_cols)

# Add the label column
df_num["label"] = y_train.values

#  Compute correlations
correlations = df_num.corr()["label"].sort_values(ascending=False)
print(correlations)


label                    1.000000
receiver_spam_ratio      0.964283
capital_ratio            0.171638
hour                     0.113581
day_of_week_Tuesday      0.060443
day_of_week_Thursday     0.054798
urls                     0.027610
day_of_week_Saturday     0.025457
day_of_week_Sunday       0.011685
day_of_week_Monday       0.010733
exclamation_count       -0.014751
day_of_week_Wednesday   -0.032130
question_count          -0.046233
capital_letter_count    -0.063771
day_of_week_Friday      -0.066303
special_char_count      -0.174104
Name: label, dtype: float64


###  Correlation
1.receiver_spam_ratio is extremely correlated with the label with 0.964! corrolation rate

This is almost a perfect linear signal.

Any model even a very simple one can latch onto this feature and predict the label almost perfectly.
That explains why precision ≈ recall ≈ F1 ≈ accuracy ≈ 99%: all models are essentially “cheating” by using this one strong signal.

---
2.Negative correlations exist but are small

special_char_count has -0.17 correlation. This can influence the model a little, but it’s minor compared to the huge receiver_spam_ratio.


### Right now, the models are basically memorizing that one feature, not learning any real patterns from the other features.

Removing it lets you see the real predictive power of the remaining features then recomputing the scores

### Even after removing the “cheating” feature, models perform very well.

#### Model Evaluation Results after removing receiver_spam_ratio 'high correlated column'

| Model                  | Accuracy | Precision | Recall  | F1-Score |
|------------------------|----------|-----------|---------|----------|
| Logistic Regression    | 0.9974   | 0.9974    | 0.9974  | 0.9974   |
| Linear SVM             | 0.9976   | 0.9976    | 0.9975  | 0.9975   |
| Naive Bayes            | 0.9635   | 0.9617    | 0.9666  | 0.9632   |


This suggests your dataset has redundant signals, or patterns are strong enough for linear models to separate spam vs. non-spam.

### Key insight: the model isn’t memorizing a single feature anymore.

Instead, it’s likely learning combinations or patterns among the remaining features:

such as capital_ratio + special_char_count + hour patterns together might separate spam from non-spam.

Logistic Regression / SVM can exploit linear combinations efficiently, which explains why their scores are still high.

Naive Bayes does worse (96%) because it assumes feature independence, so it can’t capture interactions between weak features as effectively.

### overfit & under fit or is it a good fit with these similar recall and precision metrics?

1.Overfitting : model performs much better on training data than on test data. It “memorizes” training examples.

2.Underfitting :model performs poorly on both training and test data. It cannot capture the patterns.

3.Good fit :model performs similarly on train and test, and metrics are reasonably high.

In [12]:
import joblib
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

# Load preprocessed train/test features
data = joblib.load("artifacts/feature_data.joblib")
X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]

# Drop 'receiver_spam_ratio'
numeric_cols = [
    'urls', 'receiver_spam_ratio', 'hour',
    'capital_letter_count', 'capital_ratio',
    'exclamation_count', 'question_count', 'special_char_count',
    'day_of_week_Friday', 'day_of_week_Monday',
    'day_of_week_Saturday', 'day_of_week_Sunday',
    'day_of_week_Thursday', 'day_of_week_Tuesday',
    'day_of_week_Wednesday'
]
drop_idx = numeric_cols.index('receiver_spam_ratio')
num_cols_start = X_train.shape[1] - len(numeric_cols)
abs_idx = num_cols_start + drop_idx

X_train = X_train[:, np.arange(X_train.shape[1]) != abs_idx]
X_test = X_test[:, np.arange(X_test.shape[1]) != abs_idx]

print("Dropped 'receiver_spam_ratio'")
print("New X_train shape:", X_train.shape)
print("New X_test shape:", X_test.shape)

# ----------------------
# Define and train models
# ----------------------

# Logistic Regression
log_model = LogisticRegression(max_iter=2000, random_state=42)
param_grid_log = {'C':[0.01,0.1,1,10], 'solver':['liblinear','saga']}
grid_log = GridSearchCV(log_model, param_grid_log, cv=3, scoring='accuracy', n_jobs=-1)
grid_log.fit(X_train, y_train)

# Linear SVM
svm_model = LinearSVC(max_iter=2000, random_state=42)
param_grid_svm = {'C':[0.1,1,10]}
grid_svm = GridSearchCV(svm_model, param_grid_svm, cv=3, n_jobs=-1)
grid_svm.fit(X_train, y_train)

# Naive Bayes
nb_model = MultinomialNB()
param_grid_nb = {'alpha':[0.1,0.5,1.0,2.0]}
grid_nb = GridSearchCV(nb_model, param_grid_nb, cv=3, n_jobs=-1)
grid_nb.fit(X_train, y_train)

# ----------------------
# Evaluation function
# ----------------------
def evaluate_train_test(grid_model, X_train, y_train, X_test, y_test):
    best_model = grid_model.best_estimator_
    y_pred_train = best_model.predict(X_train)
    y_pred_test  = best_model.predict(X_test)

    metrics = {
        "Dataset": ["Train", "Test"],
        "Accuracy": [
            accuracy_score(y_train, y_pred_train),
            accuracy_score(y_test, y_pred_test)
        ],
        "Precision": [
            precision_score(y_train, y_pred_train, average="macro"),
            precision_score(y_test, y_pred_test, average="macro")
        ],
        "Recall": [
            recall_score(y_train, y_pred_train, average="macro"),
            recall_score(y_test, y_pred_test, average="macro")
        ],
        "F1-Score": [
            f1_score(y_train, y_pred_train, average="macro"),
            f1_score(y_test, y_pred_test, average="macro")
        ]
    }
    return pd.DataFrame(metrics)

# Evaluate all models
df_log = evaluate_train_test(grid_log, X_train, y_train, X_test, y_test)
df_svm = evaluate_train_test(grid_svm, X_train, y_train, X_test, y_test)
df_nb  = evaluate_train_test(grid_nb, X_train, y_train, X_test, y_test)

# Combine into one table
df_log['Model'] = "Logistic Regression"
df_svm['Model'] = "Linear SVM"
df_nb['Model']  = "Naive Bayes"

df_all = pd.concat([df_log, df_svm, df_nb], ignore_index=True)
df_all = df_all[['Model', 'Dataset', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]
df_all[["Accuracy", "Precision", "Recall", "F1-Score"]] = df_all[["Accuracy", "Precision", "Recall", "F1-Score"]].round(4)

print("Train vs Test Metrics for All Models:\n")
display(df_all)


Dropped 'receiver_spam_ratio'
New X_train shape: (31311, 5014)
New X_test shape: (7828, 5014)
Train vs Test Metrics for All Models:



,Model,Dataset,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,Train,0.9990,0.9989,0.9990,0.9990
1,Logistic Regression,Test,0.9974,0.9974,0.9974,0.9974
2,Linear SVM,Train,0.9992,0.9992,0.9992,0.9992
3,Linear SVM,Test,0.9976,0.9976,0.9975,0.9975
4,Naive Bayes,Train,0.9663,0.9645,0.9692,0.9661
5,Naive Bayes,Test,0.9635,0.9617,0.9666,0.9632


# Model Evaluation and Overfit/Underfit finalizing and  Interpretation

After removing the highly correlated feature `receiver_spam_ratio` and training all models on the cleaned TF-IDF + numeric features dataset, we get the following metrics:

| Model                 | Train vs Test | Interpretation |
|-----------------------|---------------|----------------|
| Logistic Regression   | 0.9990 → 0.9974 | Very slight overfit, basically perfect fit |
| Linear SVM            | 0.9992 → 0.9976 | Very slight overfit, extremely accurate |
| Naive Bayes           | 0.9663 → 0.9635 | Well-fit, simpler model, generalizes nicely |

### Key Points:

- **Feature cleaning:** `receiver_spam_ratio` was removed due to very high correlation with the label. Other numeric features (`capital_ratio`, `hour`, `urls`, etc.) remain and provide meaningful signals.  
- **Model behavior:**
  - Logistic Regression & Linear SVM are extremely accurate and slightly overfit, but test metrics remain excellent.  
  - Naive Bayes is simpler, slightly lower scores, but generalizes well.  
- **Dataset:** Balanced classes and clean TF-IDF features help all models learn meaningful patterns rather than just memorizing labels.  
- **Conclusion:** The models are performing very well on this dataset. Given the low correlation of the remaining features, this suggests the dataset is clean, features are informative, and the models are effectively learning patterns rather than overfitting due to noise.
